# Python-1, Лекция 4.

[Оригинал](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) лекции.

Сегодня мы поговорим про ссылки, изменяемость объектов и начнем говорить про функции.

## Переменные - это не коробки!

У нас есть две переменные `a` и `b`. Мы присваиваем двум этим переменным список чисел:

In [1]:
a = [1, 2, 3]
b = [1, 2, 3]

assert a == b

In [2]:
a = [1, 2, 3]
b = a
b.append(4)
assert a == [1, 2, 3, 4]

Почему так происходит? Потому что мы не создали вторую коробку `b`, в которой хранится новый такой же список, а просто приклеили стикер `b` к уже созданному в памяти списку. Тут уже нужно быть аккуратными.

***Картинка в [оригинале](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) данной лекции.***

## Идентичность и равенство

Раньше нам уже встречались `is` и `==` (на самом деле это dunder метод __eq__).

- `is` сравнивает `id` объектов.
- `==` сравнивает объекты реализованным у них методом сравнения.

Давайте рассмотрим пример как отыскать импостера:

In [3]:
cookie = {"name": "Cookie", "color": "Grey", "size": "small"}
brownie = cookie
assert cookie is brownie

In [4]:
id(cookie), id(brownie)

(137385533874240, 137385533874240)

Как видим id cookie и brownie одинаковые. Это все о тех же стикерах на объект. Объект в итоге создался один. Конечно cookie также равен brownie:

In [5]:
assert cookie == brownie

In [6]:
cookie["age"] = 3
assert brownie["age"] == 3

In [7]:
brownie

{'name': 'Cookie', 'color': 'Grey', 'size': 'small', 'age': 3}

Опять же, по 'стикеру' cookie мы добавили словарю возраст: сам объект изменился. 'Стикер' brownie **указывает** на тот же самый объект, поэтому и в нем появился новый ключ.

Теперь добавим импостера:

In [8]:
imposter = {"name": "Cookie", "color": "Grey", "size": "small", "age": 3}
assert cookie == imposter

In [9]:
assert imposter is not cookie

In [10]:
id(cookie), id(imposter)

(137385533874240, 137385533964096)

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Равные объекты, но сами объекты в памяти уже разные, так как айди у них отличаются!
    </span>
</div>

## Когда ==, а когда is?

На самом деле чаще всего мы используем сравнение, а не идентичность.

Однако есть случаи когда правильно использовать `is`:

In [11]:
a = 1
b = None
assert b is None
assert a is not None

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        is быстрее чем == так как его нельзя переопределить
    </span>
</div>

Однако не стоит обманываться скоростью и не нужно никогда проверять на равенствно объекты через `is`.

In [12]:
a = 5
b = 5

assert a is b

id(a), id(b)

(10446536, 10446536)

<div style="
    background-color: #8B0000;
    padding: 15px;
    border: 2px dashed #ba0606;
    border-radius: 5px;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 20px;
">
<span style="color: white; font-weight: bold;">
        Антипаттерн: сравнение объектов при помощи is!
    </span>
</div>

Конечно, такое сравнение может сработать на маленьких числах, которые изначально загружаются в память при запуске программы, однако, если немного увеличим числа, то все уже поломается:

In [13]:
a = 1 << 10
b = 1 << 10
assert a is b, "Числа не равны"

AssertionError: Числа не равны

In [14]:
assert a == b
id(a), id(b)

(137385872241008, 137385872239216)

Видим что в данном примере `id` уже разные.

## Список под капотом

Нам известно, что получение элемента в списке по индексу - это очень быстрая операция. Если точнее, эта операция имеет сложность `O(1)` (О-нотацию вы разберете на курсе алгоритмов, но это самая лучшая скорость). Давайте попробуем разобраться почему это работает именно так.

Представляю вам картинку того, как что-либо лежит в памяти, например объекты в списке:

***Картинка в [оригинале](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) данной лекции.***



В списке нам известно по какому адресу лежит нулевой элемент списка. При этом нам известно, что каждый объект занимает 8 байт; Следовательно, чтобы достать элемент по индексу нам достаточно сделать:

```python
head + idx * 8
```
где `head` - адрес начала списка.

In [15]:
head = 0x00001234
for i in range(5):
    print(head, hex(head))
    head += 8

4660 0x1234
4668 0x123c
4676 0x1244
4684 0x124c
4692 0x1254


Созревает логичный вопрос: почему каждый элемент в списке весит одинаковое количество байт?

<style>
.spoiler {
  color: transparent;
  background-color: black;
}
.spoiler:hover {
  color: white;
}
</style>
<span class="spoiler">Потому что в списке мы храним указатели!</span>

In [16]:
guests = ["Frank", "Claire", "Zoe", True, 42]

***Картинка в [оригинале](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) данной лекции.***

Итак, чтобы получить значение по индексу мы делаем две операции:
1. Получить указатель на значение при помощи сдвига адреса.
2. Получить значение, разименовав указатель.

С разименованием указателей вы подробнее ознакомитесь на курсе плюсов.

Тут вступает в игру следующая проблема: список у нас динамичный и изменяемый, но мы не выделяем ему бесконечно памяти. Как тогда происходит добавление объекта в список?

Это называется амортизация - расширение вдвое размера списка, при этом длина увеличивается на одну вставку, а не вдвое.

***Картинка в [оригинале](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) данной лекции.***

Эти два механизма и делают списки самой используемый структурой.

## Относительная неизменяемость

Теперь поговорим об изменяемых и неизменяемых типах.

Бытует мнение, что кортеж - неизменяемый тип данных. Это так, но есть одно но:

In [17]:
t1 = (1, 2, [3, 4])
t2 = (1, 2, [3, 4])

In [18]:
assert t1 == t2

In [19]:
id(t1[-1]), t1[-1]

(137385533893632, [3, 4])

Можем ли мы все-таки изменить кортеж?

In [20]:
t1[-1].append(5)

<div style="
    background-color: #FFBA00;
    padding: 15px;
    border-left: 5px solid #ffcc00;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Кортеж можно изменить.
    </span>
</div>

In [21]:
t1

(1, 2, [3, 4, 5])

In [22]:
t1 == t2

False

Что произойдет с кортежем в данном случае?

In [23]:
t = (1, 2, [3, 4])
t += [5, 6]

TypeError: can only concatenate tuple (not "list") to tuple

In [24]:
t

(1, 2, [3, 4])

<div style="
    background-color: #8B0000;
    padding: 15px;
    border: 2px dashed #ba0606;
    border-radius: 5px;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 20px;
">
<span style="color: white; font-weight: bold;">
        Операция += неатомарна. Код может упасть, но исходный объект изменится.
    </span>
</div>

## Копии объектов

In [25]:
l1 = [1, [2, 3], (4, 5, 6)]
l2 = list(l1)

assert l2 == l1
assert l2 is not l1
assert l2[1] is l1[1]

Что здесь произошло?

Мы создаем новый список на основе первого, но уже делаем его копию, то есть создаем второй объект. Если изменим первый список, то второй не изменится.

In [26]:
l1[1].append(3)
l2

[1, [2, 3, 3], (4, 5, 6)]

А тут уже что-то пошло не так...

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        shallow copy - дублируется внешний контейнер, однако копия наполнена ссылками на те же объекты, что и в оригинальном контейнере.
    </span>
</div>

In [27]:
assert l1[1] is l2[1]

Как видим это правда, вложенные объекты идентичны и равны, а значит у нас не создался никакой второй вложенный список и вложенный кортеж.

<div style="
    background-color: #FFBA00;
    padding: 15px;
    border-left: 5px solid #ffcc00;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Такое копирование рекомендуется только с неизменяемыми объектами внутри для экономии памяти.
    </span>
</div>

In [28]:
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)

In [29]:
l1 = [3, [66, 55, 44], (7, 8, 9)]
l2 = list(l1)
l1.append(100)
l1[1].remove(55)
print("l1:", l1)
print("l2:", l2)
l2[1] += [33, 22]
l2[2] += (10, 11)
print("l1:", l1)
print("l2:", l2)

l1: [3, [66, 44], (7, 8, 9), 100]
l2: [3, [66, 44], (7, 8, 9)]
l1: [3, [66, 44, 33, 22], (7, 8, 9), 100]
l2: [3, [66, 44, 33, 22], (7, 8, 9, 10, 11)]


Можем наглядно посмотреть что произойдет в результате всех операцией выше. Также можем посмотреть пошагово при помощи сайта https://pythontutor.com/.

***Картинка в [оригинале](https://github.com/DanielShinoda/ami_python_25_lectures/blob/main/lectures/04.%20%D0%A1%D1%81%D1%8B%D0%BB%D0%BA%D0%B8.%20%D0%98%D0%B7%D0%BC%D0%B5%D0%BD%D1%8F%D0%B5%D0%BC%D0%BE%D1%81%D1%82%D1%8C/Lecture_04.ipynb) данной лекции.***

Давайте теперь попробуем создать матрицу (наверное вы уже знакомы с этим объектом с курса линейной алгебры).

In [30]:
a = [[0] * 5] * 5
a

[[0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0]]

In [31]:
assert a[0] is a[1]

In [32]:
a[0][1] = 2
a

[[0, 2, 0, 0, 0],
 [0, 2, 0, 0, 0],
 [0, 2, 0, 0, 0],
 [0, 2, 0, 0, 0],
 [0, 2, 0, 0, 0]]

<div style="
    background-color: #FFBA00;
    padding: 15px;
    border-left: 5px solid #ffcc00;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Таким образом создается список, внутри которого 5 ссылок на один и тот же объект.
    </span>
</div>

In [33]:
a = [[0] * 5 for _ in range(5)]
assert a[0] is not a[1]

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Создавайте вложенные объекты явно.
    </span>
</div>

## Глубокое копирование.

In [34]:
from copy import deepcopy

l1 = [1, [2, 3], (4, 5, 6)]
l2 = deepcopy(l1)

assert l2 == l1
assert l2 is not l1
assert l2[1] is not l1[1]

l1[1].append(4)

assert l1[1] == [2, 3, 4], l2[1] == [2, 3]

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        deep copy - полное копирование объектов, включая все вложенные в них объекты.
    </span>
</div>

Итак, когда что используем?

Shallow copy:

- когда работаем с простыми объектами без вложенности;
- когда нам важно экономить память и нестрашно если один объект расползется в несколько мест.

Deepcopy:

- когда работаем с вложенными данными;

Ниже посмотрим на цикличные ссылки:

In [35]:
a = [10, 20]
b = [a, 30]
a.append(b)
a, a[2], a[2][0][2], a[2][0][2][0][2]

([10, 20, [[...], 30]],
 [[10, 20, [...]], 30],
 [[10, 20, [...]], 30],
 [[10, 20, [...]], 30])

In [36]:
a = []
a.append(a)
assert a is a[0], a[0] is a[0][0]
assert a == a[0], a[0] == a[0][0]
a

[[...]]

## Занимаемая память

Ниже варианты как можно посмотреть сколько памяти занимает объект:

In [37]:
[].__sizeof__()

40

In [38]:
assert [1, 2, []].__sizeof__() == [1, 2, [1, 2, 1]].__sizeof__()
assert [1, 2, ()].__sizeof__() == [1, 2, (1)].__sizeof__()

40 байт на список, а также по 8 байт на каждый объект внутри:

In [39]:
assert [1, 2, []].__sizeof__() == 40 + 8 + 8 + 8
[1, 2, []].__sizeof__()

64

In [40]:
().__sizeof__(), [].__sizeof__()

(24, 40)

In [41]:
assert (1, 2, [3]).__sizeof__() == (1, 2, 3).__sizeof__()

## Функции

### Начало

Пора нам познакомиться с функциями. Какие-то из них вам уже знакомы, например:

In [42]:
type(print), type(input), type(type)

(builtin_function_or_method, method, type)

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Функция - логический кусок кода, который делает в точности то как он называется.
    </span>
</div>

Зачем нам нужны функции?

- упрощение кода;
- логическое структурирование кода;
- переиспользование функций;

Для начала давайте посмотрим на "чистые" функции и что их определяет:

- предсказуемость;
- отсутствие побочных эффектов;
- независимость от внешнего состояния;

Дальше на примерах посмотрим что это значит.

Ну, начнем!

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Функция - это объект.
    </span>
</div>

Поэтому с функцией можно делать следующее:

- присвоить переменной или полю в классе;
- передать в качестве аргумента функции;
- использовать в качестве возвращаемого значения функции;

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        В питоне функции принято называть в <strong>snake_case</strong> нотации.
    </span>
</div>

In [43]:
def is_even(n: int) -> bool:
    """Проверяет число на четность"""
    return n % 2 == 0  # return - ключевое слово, возвращает значение из функции

Плохая реализация этой функции:

In [44]:
def is_even_dirty(n: int) -> bool:
    """Проверяет число на четность"""
    if n % 2 == 0:
        return True
    else:
        return False

<div style="
    background-color: #8B0000;
    padding: 15px;
    border: 2px dashed #ba0606;
    border-radius: 5px;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 20px;
">
<span style="color: white; font-weight: bold;">
        Выше указан грязный код, так как много лишнего, ниже варианты улучшения:
    </span>
</div>

Чуть лучше:

In [45]:
def is_even_less_dirty(n: int) -> bool:
    """Проверяет число на четность"""
    if n % 2 == 0:
        return True
    return False


In [46]:
is_even(3)

False

In [47]:
is_even.__doc__

'Проверяет число на четность'

Теперь можем выводить документацию любой функции и читать подробно что она делает:

In [48]:
print(print.__doc__)

Prints the values to a stream, or to sys.stdout by default.

  sep
    string inserted between values, default a space.
  end
    string appended after the last value, default a newline.
  file
    a file-like object (stream); defaults to the current sys.stdout.
  flush
    whether to forcibly flush the stream.


In [49]:
type(is_even), is_even.__annotations__

(function, {'n': int, 'return': bool})

Интересный момент: так как функция - это объект, то и ключом в словаре она тоже может быть:

In [50]:
d = {is_even: "is_even function"}
d

{<function __main__.is_even(n: int) -> bool>: 'is_even function'}

Можем присвоить функцию в другую переменную:

In [51]:
f = is_even
f(4)

True

Можем даже сравнивать функции:

In [52]:
assert f is is_even, f == is_even

И так тоже можем сделать:

In [53]:
d[is_even] = f
d[f](4)

True

Заметим, что мы тут по ключу `f` обратились, который и есть ссылка на объект функции, а `f` и `is_even` равны и идентичны.

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        В питоне функция всегда возвращает значение. Если в теле функции отсутствует "return", она возвращает "None"
    </span>
</div>

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        В питоне функция может содержать более одного "return"
    </span>
</div>

In [54]:
def placeholder() -> None: ...

In [55]:
assert placeholder() is None

### Рекурсия

Рекурсия — это техника в программировании, когда функция вызывает саму себя для решения задачи.

Простая аналогия: представьте два зеркала, стоящих друг напротив друга. Они создают бесконечное отражение — это и есть рекурсия в реальной жизни. Но в программировании мы всегда должны иметь условие остановки, чтобы избежать бесконечного повторения.

Рекурсивная функция состоит из:

* Базовый случай:
    - Условие, при котором функция прекращает вызывать саму себя
    - Предотвращает бесконечную рекурсию
    - Всегда должен быть достижим
* Рекурсивный шаг:
    - Функция вызывает саму себя с измененными параметрами
    - Каждый вызов должен приближать к базовому случаю

Вернемся к нашей функции факториала, но запишем ее немного иначе для наглядности:

In [56]:
def factorial(n):
    # Базовый случай: факториал 0 и 1 равен 1
    if n <= 1:
        return 1
    # Рекурсивный шаг: n! = n * (n-1)!
    return n * factorial(n - 1)


# Такая функция выглядит чище
def factorial(n: int) -> int:
    """returns n!"""
    return 1 if n < 2 else n * factorial(n - 1)


factorial(5)

120

Наглядный пример вызовов и возвращаемых значений:

```
factorial(5)
    factorial(4)
        factorial(3)
            factorial(2)
                factorial(1) → возвращает 1
            возвращает 2 * 1 = 2
        возвращает 3 * 2 = 6
    возвращает 4 * 6 = 24
возвращает 5 * 24 = 120
```

Давайте теперь напишем функцию, в которой нет точки останова вызовов.

In [57]:
def infinite_recursion():
    infinite_recursion()


infinite_recursion()

RecursionError: maximum recursion depth exceeded

Рекурсивную функцию можно преобразовать в итеративную функцию:

In [58]:
def factorial_iterative(n):
    result = 1
    for i in range(1, n + 1):
        result *= i
    return result


def factorial_recursive(n):
    return n * factorial_recursive(n - 1) if n > 1 else 1

### Функциональное программирование

Немного функционального программирования:

In [59]:
list(
    map(
        factorial,
        range(11),
    ),
)

[1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880, 3628800]

In [60]:
a = [1, 2]
b = a[1:]
b[0] = 1
a
[[[]], {}, ()]

[[[]], {}, ()]

Что вообще делает этот код?

`map` - применяет переданную первым аргументом функцию к каждому объекту в итерируемом втором аргументе. Про итерируемый мы еще много поговорим.

`list` - строит список на основе результата 11 применений функции factorial к объектам из `range(11)`

Выше мы рассмотрели примеры функций первого порядка, но есть еще функции высшего порядка, например, тот же **map**.

<div style="
    background-color: #44944A;
    padding: 15px;
    border-radius: 10px;
    border: 1px solid #fbfbfbff;
    text-align: center;
    display: flex;
    justify-content: center;
    align-items: center;
    min-height: 15px;
">
    <span style="color: white; font-weight: bold;">
        Функция высшего порядка принимает в качестве аргумента функцию или возвращает функцию.
    </span>
</div>

Рассмотрим встроенную функцию сортировки: `sorted`.

Это функция высшего порядка, потому что в качестве ключа сортировки также принимает функцию, например:

In [61]:
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]
sorted(fruits, key=len)

['fig', 'apple', 'cherry', 'banana', 'raspberry', 'strawberry']

Можем посортировать этот список порядке обратного произношения:

In [62]:
def reverse(word: str) -> str:
    return word[::-1]


reverse("abc")

'cba'

In [63]:
sorted(fruits, key=reverse)

['banana', 'apple', 'fig', 'raspberry', 'strawberry', 'cherry']

Поговорим еще немного про функциональное программирование. В питоне есть встроенные `map` и `filter`, однако в современном питоне есть механизмы посильнее и удобнее: list comprehensions и генераторные выражения (их мы рассмотрим позже).

Давайте сравним разные подходы:

In [64]:
list(map(factorial, range(6)))

[1, 1, 2, 6, 24, 120]

In [65]:
[factorial(_) for _ in range(6)]

[1, 1, 2, 6, 24, 120]

In [66]:
list(map(factorial, filter(lambda n: n % 2, range(6))))

[1, 6, 120]

In [67]:
[factorial(n) for n in range(6) if n % 2]

[1, 6, 120]

Мне намного понятнее читать списковые вложения, нежели функциональное программирование. Рекомендуется использовать именно их, не только потому что мне удобнее, а потому что в основном люди так и пишут код, так как читаемость имеет значение.

### Анонимные функции или лямбда функции

In [68]:
fruits = ["strawberry", "fig", "apple", "cherry", "raspberry", "banana"]
sorted(fruits, key=lambda word: word[::-1])

['banana', 'apple', 'fig', 'raspberry', 'strawberry', 'cherry']

Гайд как работать с лямбда функциями:

1. Напишите комментарий, объясняющий что вообще делает эта лямбда функция.
2. Немножко подумайте об этом комментарии, придумайте название для него.
3. Измените лямбду на нормальную функцию с использованием **def** и придуманного названия.
4. Удалите комментарий и лямбда функцию.